# Gold v2 — 결함 위치 박스 표시 (MMAD 방식 위치 문항용)

gold v2 결함 이미지 139장 중 위치 정답이 있는 건 20장(InsPLAD-seg)뿐입니다. 이 노트북에서 **나머지 119장에 결함 위치 박스**를 그립니다.
그린 박스로 MMAD 방식 위치 문항(결함 이미지에만, "결함이 있다. 어디인가?", 9칸 중 4지선다, "없음" 없음)을 만듭니다. gold v2 파일은 건드리지 않습니다.

**GPU 불필요 — CPU 런타임.** 매 이미지 저장 즉시 Drive에 기록되니, 중간에 끊겨도 다시 실행하면 첫 미완료부터 이어집니다.

### 표시 규칙
- 결함을 **딱 맞게 감싸는** 박스를 그립니다. 결함이 여러 곳이면 각각 그립니다.
- 결함 종류는 화면 위에 표시됩니다 (rust / missing-cap / bird-nest).
  - rust: 녹·부식이 보이는 부분
  - missing-cap: 캡·부품이 **없어진 자리** (있어야 할 위치)
  - bird-nest: 둥지·이물질 덩어리
- 사진에서 결함 위치를 도저히 특정할 수 없으면 **"위치 표시 불가"**로 저장 (문항에서 제외되고 목록에 남음)
- 노란 점선(초안)은 expert가 의심한 위치일 뿐 정답이 아닙니다. 기본은 꺼져 있습니다 (`SHOW_DRAFT`).

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
OUT_FOLDER = "lvlm_insplad_qa/gold_v2_loc_mmad"        # 박스 JSON과 새 위치 문항이 저장될 곳
ANNOTATOR = ""                                         # 이름 또는 이니셜 (기록용)
SHOW_DRAFT = False                                     # True면 expert ROI를 노란 점선 초안으로 표시
DRAFT_EXPERT = "patchcore"
VX_FOLDER = "lvlm_insplad_results/gold_v2_vision_expert_full"   # 초안용 manifest 위치
INCLUDE_SEG_REVIEW = False                             # True면 InsPLAD-seg 박스가 있는 20장도 함께 띄워 검토
PUBLISH = True

SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색

In [ ]:
# Drive 연결 + repo clone (pip 설치·GPU 없음)
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
QA_DIR = DRIVE_ROOT / QA_FOLDER
OUT_DIR = DRIVE_ROOT / OUT_FOLDER
OUT_DIR.mkdir(parents=True, exist_ok=True)
ANNOT_FILE = OUT_DIR / "annotations.json"

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""))
    p = subprocess.run(cmd, env=env, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout[-4000:])
    if p.returncode and check:
        print(p.stderr[-4000:])
        raise RuntimeError(f"command failed ({p.returncode})")
    return p

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
sys.path.insert(0, str(REPO_DIR / "scripts"))
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())

In [ ]:
# 표시할 이미지 준비 (ZIP에서 꺼내기) + 이어하기 상태
import csv
import run_gold_baselines as base
import run_gold_v2_baselines as v2_runner
import loc_box_annotation as lba

def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
rows = v2_runner.load_gold_qa_v2(QA_DIR)
loc_rows = [json.loads(x) for x in sorted(QA_DIR.glob("localization_qa_*.jsonl"))[-1].read_text(encoding="utf-8").splitlines() if x.strip()]
targets = lba.annotation_targets(rows, loc_rows, include_seg=INCLUDE_SEG_REVIEW)
image_paths = {t["image"]: base.safe_extract(SUP_ZIP, t["image"], Path("/content/loc_images")) for t in targets}

drafts = {}
if SHOW_DRAFT:
    from PIL import Image
    manifest = DRIVE_ROOT / VX_FOLDER / "merged_evidence" / "roi_manifest_all.csv"
    sizes = {}
    for t in targets:
        with Image.open(image_paths[t["image"]]) as h:
            sizes[t["core_id"]] = h.size
    with manifest.open(newline="", encoding="utf-8-sig") as f:
        drafts = lba.draft_boxes_from_manifest(list(csv.DictReader(f)), DRAFT_EXPERT, sizes)

store = lba.AnnotationStore(ANNOT_FILE, annotator=ANNOTATOR)
session = lba.AnnotationSession(targets, image_paths, store, drafts)
from collections import Counter
print("표시 대상:", len(targets), "장", dict(Counter((t["asset_category"], t["defect_gt"]) for t in targets)))
print("진행:", session.progress())

In [ ]:
# 박스 그리기 화면 — 이 셀 출력에서 작업합니다 (Enter = 저장 후 다음)
lba.register_colab(session)

In [ ]:
# 진행 상황 확인 (언제든 실행 가능)
store = lba.AnnotationStore(ANNOT_FILE, annotator=ANNOTATOR)
done = [store.data[t["core_id"]] for t in targets if t["core_id"] in store.data]
print(f"완료 {len(done)} / {len(targets)}")
print("상태:", dict(Counter(r["status"] for r in done)))
print("박스 수:", dict(Counter(len(r["boxes_xywh"]) for r in done if r["status"] == "boxed")))
for r in done:
    if r["status"] == "unlocatable":
        print("표시 불가:", r["core_id"], r["asset_category"], r["defect_gt"], r.get("note", ""))

In [ ]:
# 새 위치 문항 만들기 — 전부 끝나면 FROZEN 세트, 덜 끝났으면 미리보기(_partial)
remaining = len(targets) - len(lba.AnnotationStore(ANNOT_FILE).data.keys() & {t["core_id"] for t in targets})
cmd = [sys.executable, REPO_DIR / "scripts/build_gold_v2_loc_mmad.py",
       "--qa-dir", QA_DIR, "--annotations", ANNOT_FILE, "--out-dir", OUT_DIR]
if remaining:
    print(f"아직 {remaining}장 남음 → 미리보기로 생성")
    cmd.append("--allow-partial")
run(cmd)
print(json.dumps(json.loads((OUT_DIR / "build_summary.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=1)[:3000])

In [ ]:
# GitHub publish (박스·문항 JSON만, 이미지는 올리지 않음) — 전부 끝난 뒤에만
summary = json.loads((OUT_DIR / "build_summary.json").read_text(encoding="utf-8"))
if PUBLISH and summary.get("quality_status") == "FROZEN":
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py",
         "--out-dir", OUT_DIR, "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_loc_mmad",
         "--files", "annotations.json", "build_summary.json", summary["file"]], env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_loc_mmad/")
else:
    print("publish 안 함:", "PUBLISH=False" if not PUBLISH else f"quality_status={summary.get('quality_status')}")